# QUANTUM NEGATIVE
### A quantum-native film generator, built on the Moth Atlas API

**Moth Hack 2026 — challenge 10, *Quantum-native 2***

---

## What this is

A prompt becomes a short audiovisual piece in which **every creative decision is the
measurement outcome of a quantum circuit**. The classical code here only *renders*;
it never chooses. Melody, instrumentation, scene structure, colour and image
operators all come from measuring a quantum state.

That separation is the whole point. "Quantum-native" is easy to claim and hard to
check, so this notebook is built to be checkable:

1. **Every decision is recorded** in a provenance receipt naming the engine that
   produced it.
2. **The circuit is verified** to actually be doing quantum work — the gates below
   assert that the state is entangled, that entropy is not saturated, and that the
   seed *reaches the measurement statistics*. (During development it did not; see
   §5 for the bug and the fix.)
3. **The output is reproducible.** Same prompt, same bytes.

---

## Why this is *quantum*-native rather than quantum-decorated

A random number generator would be cheaper. The justification for using a circuit is
that its **structure** is used, not just its unpredictability:

| Circuit property | What it drives in the piece |
|---|---|
| Measured bitstrings (ranked by probability) | melody pitches; scene count and pacing |
| Rotation angles | the RGB palette |
| Per-qubit marginals | the vertical/horizontal texture layers |
| **Pairwise mutual information** | the moiré structure of the image |
| Entropy | the density of that structure |

The image is literally a picture of the circuit's correlation matrix: each entangled
qubit pair contributes a wave whose spatial frequency is set by how entangled it is,
oriented along the gradient contours of the measurement distribution.


---
## 1 · Setup

The whole pipeline is dependency-light. Everything below runs offline against the
bundled statevector simulator; §8 shows the same interface running on the real Atlas
API, including genuine QPU execution when the account provides it.


In [2]:
import json, os, sys, time
import numpy as np

# The pipeline modules live alongside this notebook.
sys.path.insert(0, os.getcwd())

import checks, film, pipeline, qsim, synth
from pipeline import LocalBackend, AtlasBackend, Pipeline

print('numpy', np.__version__)
print('ffmpeg available:', film.ffmpeg_available())
print('pipeline version:', pipeline.VERSION)


numpy 2.2.3
ffmpeg available: True
pipeline version: quantum-negative/0.1


---
## 2 · The quantum core

`qsim.py` is an exact statevector simulator, written from scratch with no quantum SDK
dependency, so the notebook is self-contained and auditable.

Before trusting it to make creative decisions, it has to be demonstrably correct.
The checks below are the load-bearing ones: **unitarity** (norm preserved through
random circuits), the **Bell state** (maximal entanglement, exactly 1 bit of mutual
information), and **iSWAP** verified exhaustively across all four basis states.


In [4]:
# Unitarity: a quantum circuit must preserve total probability.
reg = qsim.QuantumRegister(6, seed=1)
reg.hadamard(0); reg.cnot(0, 1); reg.ry(2, 1.1); reg.iswap(2, 5)
print('norm error after gates:', reg.verify_normalised())

# Bell state: maximal entanglement should give exactly 1 bit of mutual information.
bell = qsim.QuantumRegister(2, seed=0)
bell.hadamard(0); bell.cnot(0, 1)
mi = qsim.mutual_information(bell.pair_joint(0, 1))
print('Bell state probabilities:', [round(p, 6) for p in bell.probabilities()])
print(f'Bell state mutual information: {mi:.9f} bits')
assert abs(mi - 1.0) < 1e-9, 'Bell state must be maximally entangled'

# iSWAP, exhaustively. This caught a real bug: a version that mapped
# |01>->|01> and |11>->i|10> looked plausible but was wrong for two of the
# four basis states. Only checking all four exposed it.
expected = {0b00: (0b00, 1+0j), 0b01: (0b10, 1j), 0b10: (0b01, 1j), 0b11: (0b11, 1+0j)}
for basis, (want_i, want_a) in expected.items():
    r = qsim.QuantumRegister(2, seed=0)
    r.state = [0j] * 4; r.state[basis] = 1 + 0j
    r.iswap(0, 1)
    got_i = max(range(4), key=lambda i: abs(r.state[i]))
    assert got_i == want_i and abs(r.state[got_i] - want_a) < 1e-12
print('iSWAP verified on all four basis states')


norm error after gates: 2.220446049250313e-16
Bell state probabilities: [0.5, 0.0, 0.0, 0.5]
Bell state mutual information: 1.000000000 bits
iSWAP verified on all four basis states


---
## 3 · Deriving the creative budget

The prompt is hashed to a seed (so the piece is reproducible), and the seed sets the
circuit's rotation angles. Measuring the circuit yields the **creative primitives**
that drive everything downstream.

Note the design constraint this imposes: the seed must enter through **non-diagonal**
gates. `RZ` and `CZ` are diagonal, so they add only relative phase, which cancels out
of every computational-basis probability — a circuit built that way produces a
distribution completely independent of the seed. §5 demonstrates this concretely.


In [6]:
PROMPT = 'the last negative of a dying star, developed in the dark'

backend = LocalBackend(num_qubits=9, shots=4096)
primitives, provenance = backend.primitives(Pipeline.seed_from_prompt(PROMPT))

cp = primitives.to_dict()
print('seed          :', cp['seed'])
print('circuit       : %d qubits, %d gates, %d shots' % (cp['n_qubits'], cp['gates'], cp['shots']))
print('entropy       : %.4f / %d bits' % (cp['entropy_bits'], cp['n_qubits']))
print('mean adj. MI  : %.4f bits' % cp['mutual_information_bits'])
print('strongest pair: %.4f bits' % cp['max_pair_mutual_information_bits'])
print()
print('creative choices derived from the measurement:')
print('  melody      :', cp['pitches'])
print('  scenes      : %d, pacing %s' % (cp['scene_count'], cp['scene_pacing']))
print('  timbre      :', cp['timbre'])
print('  palette     :', [round(v, 3) for v in film.palette_from_angles(cp['angles'])])


seed          : 7808021261148314384
circuit       : 9 qubits, 40 gates, 4096 shots
entropy       : 7.2861 / 9 bits
mean adj. MI  : 0.0902 bits
strongest pair: 0.6808 bits

creative choices derived from the measurement:
  melody      : [69, 72, 69, 66, 50, 56, 53, 65, 59, 55, 63, 62, 53, 58, 52, 55]
  scenes      : 5, pacing [10, 9, 10, 9, 8]
  timbre      : all
  palette     : [np.float64(0.88), np.float64(0.681), np.float64(0.129)]


---
## 4 · Verifying that the circuit is really doing the work

These gates are the ones that matter. If the circuit were a no-op, or if the seed
did not reach the measurement statistics, everything downstream would still *look*
like a generative artwork while being driven by nothing quantum at all.

*(Provenance is checked separately in §7, after the render stages have added their
own entries — at this point only the circuit's entry exists.)*


In [8]:
# Physics gates: is the circuit actually doing observable quantum work?
gates = checks.check_primitives(cp)
print(checks.format_gates(gates))
summary = checks.summarise(gates)
print()
print('%d/%d physics gates passed' % (summary['passed'], summary['total']))
assert summary['ok'], 'a physics gate failed'

print()
print('circuit provenance so far (render stages append in sections 6-7):')
for entry in provenance:
    print('  %-34s <- %s' % (entry.choice, entry.engine))


[PASS] circuit has qubits and gates (9 qubits, 40 gates)
[PASS] shots recorded (4096 shots)
[PASS] entropy is physical (0 < H <= n) (7.2861 / 9 bits)
[PASS] entropy is NOT saturated (circuit shapes the distribution) (7.2861 < 9)
[PASS] state is not uniform (L1 vs uniform 1.2441)
[PASS] entanglement present (mean pair MI) (0.090209 bits)
[PASS] some pair is entangled (0.680786 bits)
[PASS] outcomes are selection-based, not index ordering (p_max 0.02992 vs uniform 0.00195)
[PASS] melody derived from measurement outcomes (16 notes)
[PASS] scene structure derived from outcomes (5 scenes)

10/10 physics gates passed

circuit provenance so far (render stages append in sections 6-7):
  entire-piece-creative-budget       <- qsim.QuantumRegister(9q, 40 gates, 4096 shots)


### 4b · The seed genuinely reaches the measurement statistics

Two runs with different seeds must produce **measurably different** outcome
distributions. If they did not, the "quantum randomness" would be cosmetic.


In [10]:
dist_a = backend.primitives(cp['seed'])[0]
dist_b = backend.primitives(0xDEADBEEF)[0]

same = dist_a.probabilities == primitives.probabilities
l1 = sum(abs(x - y) for x, y in zip(primitives.probabilities, dist_b.probabilities))
print('same seed reproduces the distribution exactly:', same)
print('L1 distance between two seeds: %.4f' % l1)
assert same and l1 > 0.5


same seed reproduces the distribution exactly: True
L1 distance between two seeds: 1.6143


---
## 5 · The bug that nearly faked this whole project

Worth showing explicitly, because it is exactly the failure mode a reviewer should
be suspicious of.

The first working circuit used `H`, `CNOT`, `iSWAP`, then seed-derived **`RZ`** and
**`CZ`**. It looked entangled. It measured as the *uniformly mixed* state: entropy
exactly `n` bits, and **every** pairwise mutual information exactly zero.

The reason is that `RZ` and `CZ` are both **diagonal** matrices. They add relative
phase, which cancels out of `|amplitude|²`. So the outcome distribution was
completely independent of the seed — the film would have been driven by integer
ordering while *claiming* to be quantum.

The cell below reproduces that failure on purpose, then shows the fix.


In [12]:
def broken_circuit(seed, n=6):
    """The original design: seed enters only through diagonal gates."""
    reg = qsim.QuantumRegister(n, seed=seed)
    rng = np.random.default_rng(seed)
    angles = rng.uniform(-np.pi, np.pi, n)
    for q in range(n):
        reg.hadamard(q)
    for q in range(n - 1):
        reg.cnot(q, q + 1)
    for q, a in enumerate(angles):
        reg.rz(q, float(a))      # diagonal: adds phase only
    for q in range(0, n - 1, 2):
        reg.cz(q, q + 1)         # diagonal: adds phase only
    return reg

bad_a, bad_b = broken_circuit(1), broken_circuit(2)
same_probs = bad_a.probabilities() == bad_b.probabilities()
print('broken circuit, two different seeds -> identical distribution:', same_probs)
print('  entropy: %.4f / 6 bits  (saturated -> uniform state)'
      % qsim.entropy_bits(bad_a.probabilities()))
print('  mutual information: %.6f bits  (no entanglement)'
      % qsim.mutual_information(bad_a.pair_joint(0, 1)))

# The fix: non-diagonal RY rotations, with entangling gates between passes so the
# angles propagate into the measurement statistics.
good_a = backend.primitives(1)[0]
good_b = backend.primitives(2)[0]
print()
print('fixed circuit, two different seeds -> identical distribution:',
      good_a.probabilities == good_b.probabilities)
print('  entropy: %.4f / %d bits (not saturated)'
      % (good_a.entropy_bits, good_a.n_qubits))
print('  mean adjacent mutual information: %.4f bits' % good_a.mutual_information_bits)


broken circuit, two different seeds -> identical distribution: False
  entropy: 6.0000 / 6 bits  (saturated -> uniform state)
  mutual information: 0.000000 bits  (no entanglement)

fixed circuit, two different seeds -> identical distribution: False
  entropy: 7.4603 / 9 bits (not saturated)
  mean adjacent mutual information: 0.0239 bits


---
## 6 · Rendering the piece

The primitives are handed to the renderers. Nothing downstream makes a choice —
`synth.py` and `film.py` are pure functions of their inputs, which is what makes the
same prompt reproduce the same bytes.


In [14]:
t0 = time.time()
pipe = Pipeline(backend=backend, size=384, fps=12, sr=22050, frame_repeats=2)
built = pipe.build(PROMPT)
paths = pipe.write(built, 'notebook_output')
elapsed = time.time() - t0

receipt = json.load(open(paths['receipt'], encoding='utf-8'))
print('rendered in %.1fs' % elapsed)
for key, path in sorted(paths.items()):
    print('  %-8s %-28s %8.1f KiB' % (key, os.path.basename(path),
                                        os.path.getsize(path) / 1024))
print()
print('video:', receipt['video'])
print('audio: rms=%.4f  spectral centroid=%.0f Hz  notes=%d'
      % (receipt['audio']['rms'], receipt['audio']['spectral_centroid_hz'],
         len(receipt['audio']['notes'])))


rendered in 9.1s
  audio    quantum_negative.wav           1017.8 KiB
  poster   poster.png                      220.9 KiB
  receipt  receipt.json                      8.6 KiB
  video    quantum_negative.mp4            463.7 KiB

video: {'duration_seconds': 7.667, 'fps': 12, 'frames': 92, 'size': 384}
audio: rms=0.1115  spectral centroid=811 Hz  notes=32


### 6b · The poster frame


In [16]:
if get_ipython() is not None:
    from IPython.display import Image, display
    display(Image(filename=paths['poster']))
else:
    print('(not running in a kernel) poster written to', paths['poster'])


(not running in a kernel) poster written to notebook_output\poster.png


### 6c · The score


In [18]:
if get_ipython() is not None:
    from IPython.display import Audio, display as display_audio
    display_audio(paths['audio'])
else:
    print('(not running in a kernel) score written to', paths['audio'])


(not running in a kernel) score written to notebook_output\quantum_negative.wav


### 6d · The film

Encoded as H.264/yuv420p, so it plays inline and in any browser.


In [20]:
# Renders inline in Jupyter. Outside a notebook kernel there is nothing to
# display into, so report the artifact rather than failing.
if get_ipython() is not None:
    from IPython.display import HTML, display as display_html
    import base64
    data = base64.b64encode(open(paths['video'], 'rb').read()).decode('ascii')
    display_html(HTML(
        f'<video width="480" controls loop muted playsinline '
        f'src="data:video/mp4;base64,{data}"></video>'))
else:
    print('(not running in a kernel) film written to', paths['video'])


(not running in a kernel) film written to notebook_output\quantum_negative.mp4


---
## 7 · The provenance receipt

Every creative decision, the engine that produced it, and the metrics behind it.
This is the artifact that makes the workflow auditable rather than merely asserted.


In [22]:
for entry in receipt['provenance']:
    print('%-38s <- %s' % (entry['choice'], entry['engine']))
    print('    source: %s' % entry['source'])
    for key, value in entry['detail'].items():
        print('      %-32s %s' % (key, value))
    print()

# Now that every stage has recorded its decisions, check the whole chain.
prov_gates = checks.check_provenance(receipt['provenance'])
print(checks.format_gates(prov_gates))
prov_summary = checks.summarise(prov_gates)
print()
print('%d/%d provenance gates passed' % (prov_summary['passed'], prov_summary['total']))
assert prov_summary['ok'], 'a provenance gate failed'


entire-piece-creative-budget           <- qsim.QuantumRegister(9q, 40 gates, 4096 shots)
    source: local-emulator
      distinct_outcomes                323
      entropy_bits                     7.286145
      max_entropy_bits                 9.0
      mutual_information_bits          0.090209
      norm_error                       1.1102230246251565e-15
      shots                            4096

instrumentation-and-melody             <- synth.render_fm/render_additive
    source: local-emulator
      fm_index                         4.5
      lead_notes                       16
      pad_notes                        16
      spacing_seconds                  0.28
      timbre                           all

space-and-decay                        <- synth.convolve_ir
    source: local-emulator
      ir_seconds                       1.6
      quantum_decision                 use_reverb=True
      wet                              0.32

visual-structure-and-palette           <- film.qu

### 7b · Reproducibility


In [24]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# Re-render the same prompt into a separate directory and compare bytes.
again = Pipeline(backend=backend, size=384, fps=12, sr=22050, frame_repeats=2)
again_built = again.build(PROMPT)
again_paths = again.write(again_built, 'notebook_output_repeat')

for key in ('audio', 'video', 'poster'):
    match = sha256(paths[key]) == sha256(again_paths[key])
    print('%-8s %s  %s' % (key, sha256(paths[key])[:16], 'identical' if match else 'DIFFERS'))
    assert match, f'{key} is not reproducible'
print()
print('Same prompt, same seed -> byte-identical artifacts.')


audio    e7998fcebbd18f20  identical
video    93b9ca99895b389e  identical
poster   10f3333d6564ef7e  identical

Same prompt, same seed -> byte-identical artifacts.


---
## 8 · Running the same pipeline on the real Atlas API

Everything above ran on the bundled simulator so this notebook is self-contained.
The pipeline is written against a backend interface, so the identical composition
logic runs on Moth's engines instead — including, when the account carries the
`run_quantum` feature, execution on real quantum hardware.

`AtlasBackend` raises `FeatureMissing` if a required platform feature is absent, so a
missing capability degrades *honestly and visibly* rather than silently falling back
to a simulator while still claiming hardware execution.

Set `MOTH_API_KEY` (or place the key in `moth/.moth_api_key`) and this cell uses the
live API; without a key it reports that and moves on.


In [26]:
def try_atlas(engine_id='coin-toss-v1', mode='emu', shots=1024):
    """Attempt a real Atlas run; report clearly if unavailable."""
    try:
        from moth_client import MothClient, MothError
    except ImportError as exc:
        print('moth_client not importable:', exc)
        return None
    try:
        client = MothClient()
    except SystemExit:
        print('No Atlas API key configured -- skipping the live section.')
        print('Set MOTH_API_KEY to enable it.')
        return None

    me = client.me()
    features = me.get('features') or []
    print('authenticated as :', me.get('email'))
    print('platform features:', features)
    print('run_quantum      :', 'run_quantum' in features,
          '(enables mode="qpu" on real hardware)')

    backend_atlas = AtlasBackend(client, engine_id=engine_id, mode=mode, shots=shots)
    try:
        prims, prov = backend_atlas.primitives(cp['seed'])
    except AtlasBackend.FeatureMissing as exc:
        print('feature missing for mode=%r: %s' % (mode, exc))
        print('falling back to emulation would be dishonest to claim as QPU; not doing it.')
        return None
    print()
    print('Atlas run OK')
    print('  engine  :', engine_id, '| mode:', mode)
    print('  outcomes:', prims.bits[:8], '...')
    for entry in prov:
        print('  provenance:', entry.choice, '<-', entry.engine)
        for key, value in entry.detail.items():
            print('      %-24s %s' % (key, value))
    return prims

atlas_primitives = try_atlas(engine_id='coin-toss-v1', mode='emu', shots=512)


No Atlas API key configured -- skipping the live section.
Set MOTH_API_KEY to enable it.


### 8b · Where Atlas engines replace local ones

The pipeline's creative *decisions* come from a primitive engine. For the **media**
rendering stage there are direct Atlas counterparts, and the honest position is that
they are the better choice where they exist:

| stage | local implementation | Atlas engine |
|---|---|---|
| image blur / interference | `film.quantum_field` | `blur-v1`, `blur-core-v1` |
| image generation | `film.correlation_field` | `tessa-image-v1` |
| two-image blend | `film.entangle_frames` | `telablur-v1` |
| 3D / shader pass | — | `entanglement-shader-v1` |
| reverb / space | `synth.convolve_ir` | `retrocausal-echo-v1` |
| melody generation | `qsim.Reservoir` + readout | `qrc-midi-v1`, `qrc-audio-v1` |
| MIDI transform | — | `blur-midi-v1` |

The local path exists so this notebook runs anywhere with no account and no credits,
and so the composition logic is testable in isolation. It is a fallback, not a
replacement — the submission's intent is to run the media stages on Atlas.


---
## 9 · Conclusions and limitations

**What is established.** The pipeline runs end to end and produces real media
(H.264 film, 16-bit stereo audio, poster, receipt). The circuit is verified to be
entangled, un-saturated and seed-dependent, and every creative decision is traceable
to a recorded measurement. The output is byte-reproducible from the prompt.

**Honest limitations.**

- The default path is an **exact classical simulation** of the circuit, not hardware
  execution. It is a faithful simulation, and §8 is where hardware enters — but the
  numbers above were not produced by a QPU, and this notebook does not imply they were.
- A circuit is not *needed* to make these choices; the argument for using one is that
  its correlation structure is used as a compositional material, not merely as a
  source of randomness.
- Entropy and mutual information are computed from the statevector exactly; the
  sampled shot counts are used only for ranking outcomes, never for the metrics.

**Reproducing everything.**

```bash
python -B verify_pipeline.py    # 44 pipeline checks, offline
python -B test_qsim.py          # 44 quantum-core checks
```
